DATE: 20/04/2026 :: NOTE: we will be training the YOLOv8n ; [**(OPSET VALUE HAS BEEN SHIFTED TO VALUE 16, VALUE BETWEEN 15-20 is SUPPORTED)**](https://developer.aitrios.sony-semicon.com/docs/raspberry-pi-ai-camera/edge-mdt?version=1.1.0&progLang=#:~:text=OS-,3%2E2,2%2E13)

# Drone Detection — Resumable Fine-Tuning [FINAL COLAB]
**Model:** YOLOv8n | **Target:** RPi5 + IMX500  
**Runtime:** Runtime → Change runtime type → **T4 GPU**

---
## Daily workflow

### Day 1 (fresh start)
1. Set `RESUME_MODE = False` in A0
2. Run all cells
3. `last.pt` auto-downloads to your computer after every epoch

### Day 2+ (resume)
1. Set `RESUME_MODE = True` in A0
2. Run all cells — upload your latest `checkpoint_epoch_XX.pt` when prompted in A1
3. Dataset re-downloads fresh (~5 min), training resumes from your checkpoint

```
Day 1: epochs  1–5   → save checkpoint_epoch_05.pt
Day 2: epochs  6–10  → save checkpoint_epoch_10.pt
Day 3: epochs 11–15
Day 4: epochs 16–20
Day 5: epochs 21–25
Day 6: epochs 26–30  → run A5, A6, A7 → download best.onnx
```

---
## A0 — Configuration (only section you edit each session)

In [1]:
# A0 — Config
# ─────────────────────────────────────────────────────────────────
# RESUME_MODE:
#   False → Day 1: fresh start, train from yolo11n.pt
#   True  → Day 2+: upload last checkpoint, resume training
#
# EPOCHS_THIS_SESSION:
#   How many epochs to run today. 5 is safe for Colab free tier.
# ─────────────────────────────────────────────────────────────────

RESUME_MODE         = False   # ← Change to True from Day 2 onwards
EPOCHS_THIS_SESSION = 200       # ← epochs to run today

EPOCH_OFFSET = 0   # ← Change to 7 on Day 2, 14 on Day 3, etc.
# Fixed config — do not change

MODEL_NAME  = 'yolov8n_drone'
PROJECT_DIR = '/content/drone_finetune'
DATASET_DIR = '/content/drone_dataset'
YAML_PATH   = '/content/data.yaml'
WEIGHTS_DIR = f'{PROJECT_DIR}/{MODEL_NAME}/weights'
LAST_PT     = f'{WEIGHTS_DIR}/last.pt'
BEST_PT     = f'{WEIGHTS_DIR}/best.pt'
RESULTS_PNG = f'{PROJECT_DIR}/{MODEL_NAME}/results.png'

import os
os.makedirs(PROJECT_DIR, exist_ok=True)

print(f'✓ Config loaded')
print(f'  Mode         : {"RESUME" if RESUME_MODE else "FRESH START"}')
print(f'  Epochs today : {EPOCHS_THIS_SESSION}')
print(f'  Weights dir  : {WEIGHTS_DIR}')

---
## A1 — Install + GPU + Upload checkpoint (resume only)

In [2]:
# A1.1 — Install packages
!pip install ultralytics huggingface_hub --quiet
print('✓ Packages installed')

In [3]:
# A1.2 — Verify GPU
import torch
if torch.cuda.is_available():
    print(f'✓ GPU   : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    print(f'  CUDA  : {torch.version.cuda}')
else:
    print('✗ No GPU — STOP. Switch runtime to T4 GPU.')

In [4]:
# A1.3 — Upload checkpoint (RESUME MODE only)
# Day 1: skipped automatically.
# Day 2+: pick your latest checkpoint_epoch_XX.pt from Downloads when prompted.

if RESUME_MODE:
    from google.colab import files
    import shutil

    print('Select your latest checkpoint_epoch_XX.pt from your Downloads folder...')
    uploaded = files.upload()

    uploaded_filename = list(uploaded.keys())[0]
    os.makedirs(WEIGHTS_DIR, exist_ok=True)
    shutil.move(uploaded_filename, LAST_PT)

    print(f'\n✓ Checkpoint placed at: {LAST_PT}')
    print(f'  Training will resume from this checkpoint.')
else:
    print('Fresh start — skipping checkpoint upload.')

---
## A2 — Download Dataset
Downloads fresh each session directly from HuggingFace (~5 min on Colab servers).  
No Google Drive needed.

In [5]:
# A2.1 — Create dataset directory
os.makedirs(DATASET_DIR, exist_ok=True)
print(f'✓ Dataset directory: {DATASET_DIR}')

In [6]:
# A2.2 — Download from HuggingFace
# Colab server download speed is fast (~5 min) regardless of your local internet.
from huggingface_hub import snapshot_download
print('Downloading dataset...')
snapshot_download(
    repo_id='lgrzybowski/seraphim-drone-detection-dataset',
    repo_type='dataset',
    local_dir=DATASET_DIR,
    local_dir_use_symlinks=False
)
print('\n✓ Download complete')

In [7]:
# A2.3 — Extract zip batches
import zipfile, glob
zip_files = glob.glob(os.path.join(DATASET_DIR, '**', '*.zip'), recursive=True)
print(f'Found {len(zip_files)} zip files. Extracting...')
for i, zp in enumerate(sorted(zip_files)):
    with zipfile.ZipFile(zp, 'r') as zf:
        zf.extractall(os.path.dirname(zp))
    os.remove(zp)
    if (i+1) % 10 == 0:
        print(f'  {i+1}/{len(zip_files)}...')
print('\n✓ Extraction complete')

In [8]:
# A2.4 — Verify counts
for split in ['train', 'test']:
    imgs   = glob.glob(os.path.join(DATASET_DIR, split, 'images', '*.jpg'))
    labels = glob.glob(os.path.join(DATASET_DIR, split, 'labels', '*.txt'))
    ok = '✓' if len(imgs) == len(labels) and len(imgs) > 0 else '✗ MISMATCH — re-run A2.2'
    print(f'{ok}  {split:5s}  images={len(imgs):6,}  labels={len(labels):6,}')

---
## A3 — Write data.yaml

In [9]:
# A3 — Write data.yaml
import yaml
data_config = {
    'path' : DATASET_DIR,
    'train': 'train/images',
    'val'  : 'test/images',
    'nc'   : 1,
    'names': ['drone']
}
with open(YAML_PATH, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)
print('✓ data.yaml written')
with open(YAML_PATH) as f: print(f.read())

---
## A4 — Fine-Tune

**RAM fixes:**
- `cache='disk'` — preprocessed images on disk, fast reads, zero RAM cost
- `batch=32` — RAM safe (halved from 64)
- `workers=2` — Colab free tier CPU limit
- `gc.collect()` callback — force RAM flush every epoch

**Checkpoint:** downloads to your browser after **every single epoch**

In [10]:
# A4 — Training
import gc
import shutil
import torch
from datetime import datetime
from ultralytics import YOLO
from google.colab import files

def on_epoch_end(trainer):
    """
    After every epoch:
      1. GC + CUDA flush — prevents RAM leak
      2. Print timestamp — track epoch duration
      3. Download last.pt — every epoch, numbered, never overwritten
    """
    gc.collect()
    torch.cuda.empty_cache()

    # current_epoch = trainer.epoch + 1
    # # In on_epoch_end callback, change:
    # current_epoch = trainer.epoch + 1
    # to:
    current_epoch = trainer.epoch + 1 + EPOCH_OFFSET

    timestamp     = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f'  [GC]   Epoch {current_epoch:02d} complete — RAM flushed — {timestamp}')

    if not os.path.exists(LAST_PT):
        print(f'  [CKPT] ⚠ last.pt not found — skipping (training continues)')
        return

    dest = f'/content/checkpoint_epoch_{current_epoch:02d}.pt'
    shutil.copy(LAST_PT, dest)
    print(f'  [CKPT] Downloading checkpoint_epoch_{current_epoch:02d}.pt — {timestamp}')
    files.download(dest)
    print(f'  [CKPT] Check Downloads folder. Use latest .pt to resume tomorrow.')



# Load model based on mode
if RESUME_MODE:
    print(f'WEIGHT TRANSFER MODE — loading: {LAST_PT}')
    model = YOLO(LAST_PT)          # transfers learned weights
else:
    print('FRESH START — loading yolo8n.pt')
    model = YOLO('yolov8n.pt')

model.add_callback('on_fit_epoch_end', on_epoch_end)


results = model.train(
    data     = YAML_PATH,
    epochs   = EPOCHS_THIS_SESSION,
    imgsz    = 640,
    batch    = 32,
    device   = 0,
    cache    = 'disk',
    workers  = 2,
    project  = PROJECT_DIR,
    name     = MODEL_NAME,
    exist_ok = True,
    resume   = False,              # ← THIS is the fix
    patience = 15,
    save     = True,
    plots    = True,
    verbose  = True
)

# Load model based on mode
# if RESUME_MODE:
#     print(f'RESUME MODE — loading: {LAST_PT}')
#     model = YOLO(LAST_PT)
# else:
#     print('FRESH START — loading yolo11n.pt (COCO pretrained)')
#     model = YOLO('yolov8n.pt')

# model.add_callback('on_fit_epoch_end', on_epoch_end)

# results = model.train(
#     data     = YAML_PATH,
#     epochs   = EPOCHS_THIS_SESSION,
#     imgsz    = 640,
#     batch    = 32,
#     device   = 0,
#     cache    = 'disk',
#     workers  = 2,
#     project  = PROJECT_DIR,
#     name     = MODEL_NAME,
#     exist_ok = True,
#     resume   = RESUME_MODE,
#     patience = 15,
#     save     = True,
#     plots    = True,
#     verbose  = True
# )

# print(f'\n✓ Session complete — {EPOCHS_THIS_SESSION} epochs trained')
# print(f'  → Tomorrow: RESUME_MODE=True, upload latest checkpoint_epoch_XX.pt')

---
## A5 — Validate (final session only, after epoch 30)
Target: mAP@0.5 ≥ 0.90

In [11]:
# A5 — Validate
from ultralytics import YOLO
best_model = YOLO(BEST_PT)
metrics    = best_model.val(data=YAML_PATH, split='val', device=0)
map50      = metrics.box.map50
map5095    = metrics.box.map
print(f'\n✓ Validation complete')
print(f'  mAP@0.5      : {map50:.4f}')
print(f'  mAP@0.5:0.95 : {map5095:.4f}')
if map50 >= 0.90:
    print('  ✓ Excellent — proceed to A6')
elif map50 >= 0.70:
    print('  ✓ Target met — proceed to A6')
else:
    print('  ⚠ Consider more epochs before exporting')

---
## A6 — Export to ONNX (final session only)

the following fact has been debunked: [here](https://developer.aitrios.sony-semicon.com/docs/raspberry-pi-ai-camera/edge-mdt?version=1.1.0&progLang=#_framework_extensions:~:text=OS-,3%2E2,2%2E13)


```      
`opset=12` — do not change. Sony Edge-MDT has issues with opset 13+.
```

thats why i have changed the opset value to 16 (date: 20th april 2026)

In [12]:
# A6 — Export to ONNX
from ultralytics import YOLO
best_model = YOLO(BEST_PT)
best_model.export(format='onnx', imgsz=640, simplify=True, opset=16)
print(f'✓ Export complete: {BEST_PT.replace(".pt", ".onnx")}')

---
## A7 — Download Final Outputs (final session only)

In [18]:
# A7 — Download final outputs
from google.colab import files
import shutil
shutil.make_archive('/content/drone_weights_final', 'zip', WEIGHTS_DIR)
files.download('/content/drone_weights_final.zip')  # best.onnx + best.pt
files.download(RESULTS_PNG)                         # training curves
print('✓ Downloading — keep best.onnx for Phase B (Sony IMX500 converter)')

---
## Quick Reference
```
Every session:
  A0: set RESUME_MODE and EPOCHS_THIS_SESSION
  A1.3: upload checkpoint if RESUME_MODE=True
  Run All
  Save the downloaded checkpoint_epoch_XX.pt

Final session only (after epoch 30):
  Run A5 → A6 → A7

Which file to upload each day:
  Always the highest-numbered checkpoint_epoch_XX.pt in your Downloads
```

In [17]:
from google.colab import files
files.download('/content/drone_finetune/yolo8_drone/weights/best.onnx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
from google.colab import files
# files.download('/content/checkpoint_epoch_05.pt')
for i in range(EPOCH_OFFSET + 1, EPOCH_OFFSET + 1 + EPOCHS_THIS_SESSION):
  files.download(f'/content/checkpoint_epoch_{i}.pt')
files.download(BEST_PT)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
import zipfile
import os

In [21]:
# Create a zip file
zip_filename = "checkpoints.zip"
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for i in range(EPOCH_OFFSET + 1, EPOCH_OFFSET + 1 + EPOCHS_THIS_SESSION):
        checkpoint_file = f'/content/checkpoint_epoch_{i}.pt'
        if os.path.exists(checkpoint_file):
            zipf.write(checkpoint_file, os.path.basename(checkpoint_file))
        else:
            print(f"Warning: {checkpoint_file} does not exist.")

In [22]:
from google.colab import files
files.download(zip_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>